# Brain Tumour MRI Classification — Part 1: Baseline
**Unit:** CIS143-6 Applications of AI  
**Dataset:** Brain Tumour MRI (Kaggle — masoudnickparvar)  
**Classes:** Glioma · Meningioma · No Tumour · Pituitary  
> Run on Google Colab with GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

## 1. Environment Setup

In [ ]:
!pip install -q kagglehub

import os
import random
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

print("TensorFlow:", tf.__version__)

In [ ]:
# Confirm GPU is available
gpus = tf.config.list_physical_devices('GPU')
print("GPU available:", gpus if gpus else "NONE — enable GPU in Runtime settings")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/brain_tumour_mri')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print("Drive folder ready:", DRIVE_DIR)

## 2. Dataset Download & Organisation

In [ ]:
# --- Kaggle credentials (choose ONE option) ---

# Option A: upload kaggle.json now (one-off interactive upload)
# from google.colab import files
# files.upload()   # select your kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Option B: kaggle.json already saved in your Google Drive root (recommended)
os.environ['KAGGLE_CONFIG_DIR'] = '/content/drive/MyDrive'
print("Kaggle config dir set to:", os.environ['KAGGLE_CONFIG_DIR'])

In [ ]:
import kagglehub

DATA_DIR = Path('/content/brain_tumour_data')

if not DATA_DIR.exists():
    print("Downloading dataset from Kaggle...")
    raw_path = Path(kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset"))
    shutil.copytree(raw_path, DATA_DIR)
    # Save a copy to Drive so future sessions skip the download
    drive_dataset = DRIVE_DIR / 'dataset'
    if not drive_dataset.exists():
        shutil.copytree(raw_path, drive_dataset)
        print("Persisted to Google Drive.")
    print("Done.")
else:
    print("Dataset already cached at", DATA_DIR)

print("Data directory:", DATA_DIR)

In [ ]:
CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
TRAIN_DIR = DATA_DIR / 'Training'

# Count images per class in the Training folder
counts = {}
for cls in CLASSES:
    cls_dir = TRAIN_DIR / cls
    counts[cls] = len(list(cls_dir.glob('*.jpg')))

count_df = pd.DataFrame.from_dict(counts, orient='index', columns=['Images'])
count_df['Percentage'] = (count_df['Images'] / count_df['Images'].sum() * 100).round(1)
print(count_df)
print(f"\nTotal training images: {count_df['Images'].sum()}")

In [ ]:
colours = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(counts.keys(), counts.values(), color=colours, edgecolor='white', linewidth=0.8)
ax.bar_label(bars, fmt='%d', padding=3, fontsize=10)
ax.set_title('Class Distribution — Training Folder', fontsize=13, pad=12)
ax.set_ylabel('Number of Images')
ax.set_ylim(0, max(counts.values()) * 1.15)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, cls, colour in zip(axes, CLASSES, colours):
    img_path = next((TRAIN_DIR / cls).glob('*.jpg'))
    ax.imshow(Image.open(img_path), cmap='gray')
    ax.set_title(cls.capitalize(), fontsize=12, color='white',
                 fontweight='bold', pad=6,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor=colour, alpha=0.85))
    ax.axis('off')
plt.suptitle('Sample MRI Images by Class', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Data Preparation

In [ ]:
# Build a flat dataframe of all file paths and labels from Training/
records = []
for cls in CLASSES:
    for img_path in (TRAIN_DIR / cls).glob('*.jpg'):
        records.append({'filepath': str(img_path), 'label': cls})

df = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Total images: {len(df)}")
print(df['label'].value_counts())

In [ ]:
# Stratified 70 / 15 / 15 split
# Splitting from the Training folder ourselves ensures full control and no leakage
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")
print("\nPer-class split counts:")
split_summary = pd.DataFrame({
    'Train': train_df['label'].value_counts(),
    'Val':   val_df['label'].value_counts(),
    'Test':  test_df['label'].value_counts(),
})
print(split_summary)

# Sanity check: no file appears in more than one split
overlap = set(train_df['filepath']) & set(val_df['filepath']) & set(test_df['filepath'])
print(f"\nFile overlap across splits: {len(overlap)} (expected 0)")

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Augmentation only on training data — val/test use rescale only (no leakage)
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    zoom_range=0.10,
    horizontal_flip=True,
    width_shift_range=0.10,
    height_shift_range=0.10,
)
val_test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True, seed=42)

val_gen = val_test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False)

test_gen = val_test_datagen.flow_from_dataframe(
    test_df, x_col='filepath', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False)

CLASS_NAMES = list(train_gen.class_indices.keys())
print("Class index mapping:", train_gen.class_indices)

## 4. Baseline CNN Model

In [ ]:
def build_baseline_cnn(input_shape=(224, 224, 3), num_classes=4):
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D(2, 2),
        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        # Classifier head
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dense(num_classes, activation='softmax'),
    ], name='baseline_cnn')

    model.compile(
        optimizer=tf.keras.optimizers.Adamax(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

baseline_model = build_baseline_cnn()
baseline_model.summary()

In [ ]:
EPOCHS = 20

history = baseline_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    verbose=1,
)

In [ ]:
epochs_range = range(1, EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, history.history['accuracy'], label='Train', linewidth=2)
ax1.plot(epochs_range, history.history['val_accuracy'], label='Validation',
         linewidth=2, linestyle='--')
ax1.set_title('Accuracy per Epoch', fontsize=12)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.set_ylim(0, 1.05)
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history.history['loss'], label='Train', linewidth=2)
ax2.plot(epochs_range, history.history['val_loss'], label='Validation',
         linewidth=2, linestyle='--')
ax2.set_title('Loss per Epoch', fontsize=12)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Baseline CNN — Training History', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Evaluation

In [ ]:
test_loss, test_acc = baseline_model.evaluate(test_gen, verbose=0)
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Loss     : {test_loss:.4f}")

# Generate predictions
test_gen.reset()
y_pred_probs = baseline_model.predict(test_gen, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, colorbar=True, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix — Baseline CNN', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

## 6. Error Analysis

In [ ]:
filepaths = test_gen.filepaths

misclassified = [
    (filepaths[i], y_true[i], y_pred[i], y_pred_probs[i].max())
    for i in range(len(y_true))
    if y_true[i] != y_pred[i]
]
print(f"Total misclassified: {len(misclassified)} / {len(y_true)} "
      f"({len(misclassified)/len(y_true)*100:.1f}%)")

samples = random.sample(misclassified, min(6, len(misclassified)))

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, (path, true_idx, pred_idx, conf) in zip(axes.flat, samples):
    img = Image.open(path).resize((224, 224))
    ax.imshow(img, cmap='gray')
    ax.set_title(
        f"True:  {CLASS_NAMES[true_idx]}\nPred: {CLASS_NAMES[pred_idx]}  ({conf:.1%})",
        fontsize=9, color='firebrick', fontweight='bold'
    )
    ax.axis('off')

plt.suptitle('Misclassified Samples — Baseline CNN', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Error Analysis Observations

- **Glioma ↔ Meningioma confusion**: Both tumour types produce irregular, bright regions in T1/T2 MRI scans. Without fine-grained spatial features, the baseline CNN conflates them.
- **Meningioma underperformance**: Meningioma has the fewest training samples; the model sees fewer examples and generalises poorly to its visual variants.
- **High-confidence errors**: Several misclassifications show >80% confidence, suggesting the model is overfit to superficial texture patterns rather than clinically meaningful features.
- **No-tumour false positives**: Scan artefacts (skull edge brightness, patient positioning) can resemble tumour-like bright spots, causing the model to predict a tumour class incorrectly.

In [ ]:
# Save model weights to Google Drive for reuse in later parts
save_path = str(DRIVE_DIR / 'baseline_cnn.keras')
baseline_model.save(save_path)
print("Model saved to:", save_path)